In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_interactions.csv
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/ingr_map.pkl
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_recipes.csv
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_recipes.csv
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/interactions_train.csv
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/interactions_test.csv
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_users.csv
/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/interactions_validation.csv


In [61]:
import pandas as pd
import ast
import torch
import random
import torch.nn as nn
import torch.optim as optim
import math
from torch.utils.data import Dataset, DataLoader

In [18]:
raw = "/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_recipes.csv"
df = pd.read_csv(raw)
df = df[["name", "ingredients"]]
df.head(5)

,name,ingredients
0,arriba baked winter squash mexican style,"['winter squash', 'mexican seasoning', 'mixed ..."
1,a bit different breakfast pizza,"['prepared pizza crust', 'sausage patty', 'egg..."
2,all in the kitchen chili,"['ground beef', 'yellow onions', 'diced tomato..."
3,alouette potatoes,"['spreadable cheese with garlic and herbs', 'n..."
4,amish tomato ketchup for canning,"['tomato juice', 'apple cider vinegar', 'sugar..."


In [55]:
# # Collecting ingredients and make Tokenizer
# # each cell is list, but everything is in string
# ingredients = df["ingredients"]
# table = str.maketrans("", "", "[]()''")
# ingredients_list = []
# for i in range(ingredients.size):
#     row = ingredients[i].translate(table)
#     row = row.split(",")
#     for j in row:
#         if j not in ingredients_list:
#             ingredients_list.append(j)
        

# print(ingredients_list[:5])

# id2ing = {i: char for i, char in enumerate(ingredients_list)}
# ing2id = {char: i for i, char in enumerate(ingredients_list)}

# print(id2ing[0])
# print(ing2id["winter squash"])


cleaned_recipes = df["ingredients"].apply(ast.literal_eval)

unique_ingredients = set()
for recipe in cleaned_recipes:
    for ing in recipe:
        # remove space
        unique_ingredients.add(ing.strip())

# tokens for PAD MASK UNK
special_tokens = ["[PAD]", "[MASK]", "[UNK]"]
vocab_list = special_tokens + sorted(list(unique_ingredients))

# dictonary
ing2id = {ing: i for i, ing in enumerate(vocab_list)}
id2ing = {i: ing for i, ing in enumerate(vocab_list)}

print(f"総食材数（語彙数）: {len(vocab_list)}")
print("冬カボチャのID:", ing2id.get("winter squash"))

総食材数（語彙数）: 14945
冬カボチャのID: 14771


In [57]:
class RecipeDataset(Dataset):
    def __init__(self, recipes, ing2id, max_len=16):
        self.recipes = recipes
        self.ing2id = ing2id
        self.max_len = max_len
        
        # 特殊トークンのIDを取得
        self.pad_id = ing2id["[PAD]"]
        self.mask_id = ing2id["[MASK]"]
        self.unk_id = ing2id["[UNK]"]

    def __len__(self):
        return len(self.recipes)

    def __getitem__(self, idx):
        recipe = self.recipes[idx]
        
        # テキストをIDに変換（辞書にない未知の食材は [UNK] にする）
        ing_ids = [self.ing2id.get(ing, self.unk_id) for ing in recipe]
        
        # レシピが空の場合の安全対策
        if len(ing_ids) == 0:
            ing_ids = [self.unk_id]
            
        # --- 穴埋めクイズ（MLM）の作成 ---
        # 隠す食材をランダムに1つ選ぶ
        mask_idx = random.randint(0, len(ing_ids) - 1)
        label_id = ing_ids[mask_idx] # 正解をメモしておく
        
        masked_ing_ids = ing_ids.copy()
        masked_ing_ids[mask_idx] = self.mask_id # 選んだ食材を目隠し
        
        # --- パディング（長さを max_len に揃える） ---
        if len(masked_ing_ids) < self.max_len:
            pad_length = self.max_len - len(masked_ing_ids)
            masked_ing_ids.extend([self.pad_id] * pad_length)
        else:
            # max_lenより長い場合は切り捨てる
            masked_ing_ids = masked_ing_ids[:self.max_len]
            
        return {
            "input_ingredients": torch.tensor(masked_ing_ids, dtype=torch.long),
            "label": torch.tensor(label_id, dtype=torch.long)
        }

# DatasetとDataLoaderの準備
dataset = RecipeDataset(cleaned_recipes, ing2id, max_len=16)
# バッチサイズ（1回に処理するレシピの数）はPC/Colabの性能に合わせて調整
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print("✅ Dataset & DataLoader 準備完了！")

✅ Dataset & DataLoader 準備完了！


In [62]:
class RecipeCompositionEncoder(nn.Module):
    def __init__(self, ing_vocab_size, d_model=128, nhead=4, num_layers=2, max_len=32):
        super().__init__()
        self.d_model = d_model
        
        # 1. 埋め込み層（Embedding）: 食材の背番号をベクトルに変換
        self.ing_embedding = nn.Embedding(ing_vocab_size, d_model)
        
        # 位置エンコーディング（レシピ内の順番を記憶）
        self.pos_embedding = nn.Embedding(max_len, d_model)
        
        # 2. Transformerエンコーダー（味のケミストリーを計算）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, 
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. 出力層（MLM用）: 混ざり合ったベクトルから元の材料IDを予測
        self.mlm_head = nn.Linear(d_model, ing_vocab_size)
        
        # パラメータの初期化
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ingredients):
        """
        input_ingredients: [batch_size, seq_len] (材料のID列のみを受け取る)
        """
        batch_size, seq_len = input_ingredients.size()
        device = input_ingredients.device
        
        # 位置IDを自動生成（[0, 1, 2, ..., seq_len-1]）
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
        
        # 1. 各IDをベクトル（d_model次元）に変換
        ing_emb = self.ing_embedding(input_ingredients)
        pos_emb = self.pos_embedding(positions)
        
        # 2. 融合（Fusion）: 食材ベクトルと位置ベクトルのみを足し合わせる
        x = ing_emb + pos_emb
        
        # スケーリング
        x = x * math.sqrt(self.d_model)
        
        # 3. 鍋（口の中）で混ぜ合わせる
        memory = self.transformer(x)
        
        # 4. 元の材料は何かを予測
        logits = self.mlm_head(memory)
        
        return logits

if __name__ == "__main__":
    # ─── 動作テスト ───
    print("🧠 食材特化型Encoderのテストを開始します...")
    
    DUMMY_ING_VOCAB_SIZE = 100
    BATCH_SIZE = 2
    SEQ_LEN = 5
    
    # モデルのインスタンス化（state_vocab_sizeが不要になりました）
    model = RecipeCompositionEncoder(
        ing_vocab_size=DUMMY_ING_VOCAB_SIZE, 
        d_model=64
    )
    
    # ダミーの入力データ
    dummy_ingredients = torch.tensor([
        [1, 5, 12, 0, 0],
        [8, 1, 3, 44, 0]
    ])
    
    # 推論の実行（入力が1つだけになり、非常にスッキリしました）
    output_logits = model(dummy_ingredients)
    
    print("✅ 実行完了！")
    print(f"入力テンソルのサイズ: {dummy_ingredients.shape} (Batch, SeqLen)")
    print(f"出力テンソルのサイズ: {output_logits.shape} (Batch, SeqLen, VocabSize)")

🧠 食材特化型Encoderのテストを開始します...
✅ 実行完了！
入力テンソルのサイズ: torch.Size([2, 5]) (Batch, SeqLen)
出力テンソルのサイズ: torch.Size([2, 5, 100]) (Batch, SeqLen, VocabSize)


In [63]:
# 1. デバイスの設定（GPUが使えるなら使う）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

# 2. モデルの初期化（語彙数はさっき作った vocab_list の長さ）
vocab_size = len(ing2id)
model = RecipeCompositionEncoder(
    ing_vocab_size=vocab_size, 
    d_model=128, 
    nhead=4, 
    num_layers=2, 
    max_len=16
).to(device)

# 3. 損失関数と最適化手法（オプティマイザ）
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 4. 学習ループ
num_epochs = 3  # まずは3周くらいでテスト

print("🔥 学習スタート！")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(dataloader):
        # データをGPU（またはCPU）に送る
        inputs = batch["input_ingredients"].to(device)
        labels = batch["label"].to(device)
        
        # 勾配をリセット
        optimizer.zero_grad()
        
        # 推論（順伝播）
        outputs = model(inputs)
        
        # Maskされた位置の予測結果だけを取り出してLossを計算したいですが、
        # 今回はシンプルに「出力全体（sequenceの各位置）の平均的なプーリング」か
        # 最初のトークンを使って予測する簡易版にしています。
        # 最も簡単な方法として、出力されたテンソルの「平均ベクトル」から予測させます。
        pooled_output = outputs.mean(dim=1) # [batch_size, vocab_size]
        
        # 誤差の計算
        loss = criterion(pooled_output, labels)
        
        # 逆伝播（AIの脳をアップデート）
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # 100バッチごとに進捗を表示
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}/{len(dataloader)}], Loss: {loss.item():.4f}")
            
    avg_loss = total_loss / len(dataloader)
    print(f"🌟 Epoch {epoch+1} 完了 | 平均Loss: {avg_loss:.4f}\n")

使用するデバイス: cpu
🔥 学習スタート！
Epoch [1/3], Batch [0/7239], Loss: 9.6294
Epoch [1/3], Batch [100/7239], Loss: 8.3875
Epoch [1/3], Batch [200/7239], Loss: 7.6249
Epoch [1/3], Batch [300/7239], Loss: 7.2666
Epoch [1/3], Batch [400/7239], Loss: 6.5215
Epoch [1/3], Batch [500/7239], Loss: 7.1373
Epoch [1/3], Batch [600/7239], Loss: 6.5109
Epoch [1/3], Batch [700/7239], Loss: 7.5452
Epoch [1/3], Batch [800/7239], Loss: 6.9426
Epoch [1/3], Batch [900/7239], Loss: 7.7760
Epoch [1/3], Batch [1000/7239], Loss: 6.1563
Epoch [1/3], Batch [1100/7239], Loss: 6.1971
Epoch [1/3], Batch [1200/7239], Loss: 6.3841
Epoch [1/3], Batch [1300/7239], Loss: 6.7174
Epoch [1/3], Batch [1400/7239], Loss: 6.4856
Epoch [1/3], Batch [1500/7239], Loss: 6.9576
Epoch [1/3], Batch [1600/7239], Loss: 6.5374
Epoch [1/3], Batch [1700/7239], Loss: 7.0715
Epoch [1/3], Batch [1800/7239], Loss: 7.1946
Epoch [1/3], Batch [1900/7239], Loss: 6.0321
Epoch [1/3], Batch [2000/7239], Loss: 6.9859
Epoch [1/3], Batch [2100/7239], Loss: 7.228

In [65]:
def suggest_next_ingredient(recipe_words, top_k=5):
    for _ in recipe_words:
        if _ not in unique_ingredients:
            print("the ingredient doesn't exists in ingredients list")
    model.eval() # 推論モードに切り替え
    
    # 1. 入力されたテキストをIDに変換
    ing_ids = [ing2id.get(w, ing2id["[UNK]"]) for w in recipe_words]
    
    # 2. 最後に [MASK] を追加して「次は何？」とAIに問う
    ing_ids.append(ing2id["[MASK]"])
    
    # 3. パディング（長さを16に揃える）
    while len(ing_ids) < 16:
        ing_ids.append(ing2id["[PAD]"])
    ing_ids = ing_ids[:16] # 念のため切り詰め
    
    # テンソル化してGPU/CPUへ
    input_tensor = torch.tensor([ing_ids], dtype=torch.long).to(device)
    
    # 4. 推論の実行
    with torch.no_grad():
        outputs = model(input_tensor)
        # 学習時と同じように平均プーリング
        pooled_output = outputs.mean(dim=1)
        
        # 確率（パーセンテージ）に変換して上位を取得
        probs = torch.softmax(pooled_output, dim=-1)
        top_probs, top_indices = torch.topk(probs, top_k)
        
    print(f"current recipe: {recipe_words}")
    print("Suggested ingredients Top 5:")
    for i in range(top_k):
        prob = top_probs[0][i].item() * 100
        ing_name = id2ing[top_indices[0][i].item()]
        print(f"  {i+1}. {ing_name} (prob: {prob:.1f}%)")
    print("-" * 30)

# 試しにいくつか実行してみる
suggest_next_ingredient(["tomato", "garlic", "olive oil"])
suggest_next_ingredient(["milk", "sugar", "vanilla"])
suggest_next_ingredient(["beef", "onion", "carrot"])

the ingredient doesn't exists in ingredients list
current recipe: ['tomato', 'garlic', 'olive oil']
Suggested ingredients Top 5:
  1. salt and pepper (prob: 11.6%)
  2. salt (prob: 11.0%)
  3. salt & freshly ground black pepper (prob: 3.2%)
  4. parmesan cheese (prob: 3.1%)
  5. garlic cloves (prob: 2.8%)
------------------------------
current recipe: ['milk', 'sugar', 'vanilla']
Suggested ingredients Top 5:
  1. butter (prob: 6.5%)
  2. milk (prob: 5.0%)
  3. eggs (prob: 4.1%)
  4. salt (prob: 3.8%)
  5. water (prob: 2.9%)
------------------------------
current recipe: ['beef', 'onion', 'carrot']
Suggested ingredients Top 5:
  1. onion (prob: 2.5%)
  2. garlic cloves (prob: 2.0%)
  3. tomatoes (prob: 1.4%)
  4. garlic (prob: 1.3%)
  5. green pepper (prob: 1.3%)
------------------------------
